# Notebook 01: Dataset Preparation
**Multi-Camera Tracking System — Kaggle Training Pipeline**

This notebook prepares the Market-1501 (person) and VeRi-776 (vehicle) datasets for ReID training.

**Runtime**: CPU (no GPU needed) | **Duration**: ~30 minutes

## Steps
1. Download and extract datasets from Kaggle
2. Parse dataset structures
3. Create unified CSV manifests (train/query/gallery)
4. Compute and visualize dataset statistics

In [ ]:
!pip install -q pandas matplotlib seaborn tqdm

## 1. Setup and Configuration

In [ ]:
import os
import re
import csv
import json
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Configuration — paths match Kaggle dataset mount points
MARKET1501_ROOT = Path("/kaggle/input/market-1501")
VERI776_ROOT = Path("/kaggle/input/veri-vehicle-re-identification-dataset")
OUTPUT_DIR = Path("/kaggle/working/manifests")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 2. Market-1501 Dataset Preparation

### 2.1 Dataset Structure
Market-1501 contains 1,501 person identities captured from 6 cameras.
- **Filename format**: `XXXX_cYsZ_NNNNNN_NN.jpg`
  - XXXX = person ID (-1 for junk, 0000 for distractor)
  - Y = camera ID (1-6)
  - s/Z = sequence/frame numbers

In [ ]:
def parse_market1501(root: Path):
    """Parse Market-1501 dataset into structured records."""
    splits = {
        "train": "bounding_box_train",
        "query": "query",
        "gallery": "bounding_box_test",
    }

    pattern = re.compile(r"(-?\d+)_c(\d+)s\d+_\d+_\d+\.jpg")
    all_data = {}

    for split_name, folder_name in splits.items():
        folder = root / folder_name
        if not folder.exists():
            # Try alternative paths common on Kaggle
            alt_folder = root / "Market-1501-v15.09.15" / folder_name
            if alt_folder.exists():
                folder = alt_folder
            else:
                print(f"  Warning: {folder} not found, skipping {split_name}")
                continue

        rows = []
        for img_path in tqdm(sorted(folder.glob("*.jpg")), desc=f"Parsing {split_name}"):
            match = pattern.match(img_path.name)
            if not match:
                continue
            pid = int(match.group(1))
            cam_id = int(match.group(2))
            if pid < 0:  # skip junk images
                continue
            rows.append({
                "image_path": str(img_path),
                "identity_id": pid,
                "camera_id": cam_id,
                "filename": img_path.name,
            })

        all_data[split_name] = pd.DataFrame(rows)
        print(f"  {split_name}: {len(rows)} images, {len(set(r['identity_id'] for r in rows))} IDs")

    return all_data

market_data = parse_market1501(MARKET1501_ROOT)

In [ ]:
# Save manifests as CSV
for split_name, df in market_data.items():
    if df.empty:
        continue
    manifest_path = OUTPUT_DIR / f"market1501_{split_name}.csv"
    df[["image_path", "identity_id", "camera_id"]].to_csv(manifest_path, index=False)
    print(f"Saved {manifest_path} ({len(df)} rows)")

### 2.2 Market-1501 Statistics

In [ ]:
def plot_dataset_stats(data_dict, dataset_name):
    """Plot dataset statistics: ID distribution, camera distribution, samples per ID."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{dataset_name} Dataset Statistics", fontsize=14, fontweight="bold")

    # Split sizes
    split_sizes = {k: len(v) for k, v in data_dict.items() if not v.empty}
    axes[0].bar(split_sizes.keys(), split_sizes.values(), color=["#2196F3", "#FF9800", "#4CAF50"])
    axes[0].set_title("Images per Split")
    axes[0].set_ylabel("Count")
    for i, (k, v) in enumerate(split_sizes.items()):
        axes[0].text(i, v + 100, str(v), ha="center", fontweight="bold")

    # Camera distribution (train split)
    if "train" in data_dict and not data_dict["train"].empty:
        cam_counts = data_dict["train"]["camera_id"].value_counts().sort_index()
        axes[1].bar(cam_counts.index.astype(str), cam_counts.values, color="#9C27B0")
        axes[1].set_title("Camera Distribution (Train)")
        axes[1].set_xlabel("Camera ID")
        axes[1].set_ylabel("Count")

    # Samples per identity (train split)
    if "train" in data_dict and not data_dict["train"].empty:
        id_counts = data_dict["train"]["identity_id"].value_counts()
        axes[2].hist(id_counts.values, bins=50, color="#E91E63", edgecolor="black", alpha=0.7)
        axes[2].set_title("Samples per Identity (Train)")
        axes[2].set_xlabel("Number of Images")
        axes[2].set_ylabel("Number of Identities")
        axes[2].axvline(id_counts.mean(), color="red", linestyle="--", label=f"Mean: {id_counts.mean():.1f}")
        axes[2].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{dataset_name.lower().replace('-', '')}_stats.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_dataset_stats(market_data, "Market-1501")

## 3. VeRi-776 Dataset Preparation

### 3.1 Dataset Structure
VeRi-776 contains 776 vehicle identities captured from 20 cameras.
- **Filename format**: `XXXX_cYYY_NNNNN.jpg` (or similar variants)
  - XXXX = vehicle ID
  - YYY = camera ID

In [ ]:
def parse_veri776(root: Path):
    """Parse VeRi-776 dataset into structured records."""
    splits = {
        "train": "image_train",
        "query": "image_query",
        "gallery": "image_test",
    }

    pattern = re.compile(r"(\d+)_c(\d+)_\d+.*\.jpg")
    all_data = {}

    for split_name, folder_name in splits.items():
        folder = root / folder_name
        if not folder.exists():
            for alt in [root / "VeRi", root / "VeRi-776"]:
                alt_folder = alt / folder_name
                if alt_folder.exists():
                    folder = alt_folder
                    break
            else:
                if not folder.exists():
                    print(f"  Warning: {folder} not found, skipping {split_name}")
                    continue

        rows = []
        for img_path in tqdm(sorted(folder.glob("*.jpg")), desc=f"Parsing {split_name}"):
            match = pattern.match(img_path.name)
            if not match:
                continue
            vid = int(match.group(1))
            cam_id = int(match.group(2))
            rows.append({
                "image_path": str(img_path),
                "identity_id": vid,
                "camera_id": cam_id,
                "filename": img_path.name,
            })

        all_data[split_name] = pd.DataFrame(rows)
        print(f"  {split_name}: {len(rows)} images, {len(set(r['identity_id'] for r in rows))} IDs")

    return all_data

veri_data = parse_veri776(VERI776_ROOT)

In [ ]:
for split_name, df in veri_data.items():
    if df.empty:
        continue
    manifest_path = OUTPUT_DIR / f"veri776_{split_name}.csv"
    df[["image_path", "identity_id", "camera_id"]].to_csv(manifest_path, index=False)
    print(f"Saved {manifest_path} ({len(df)} rows)")

### 3.2 VeRi-776 Statistics

In [ ]:
plot_dataset_stats(veri_data, "VeRi-776")

## 4. Cross-Dataset Summary

In [ ]:
def summarize_datasets(datasets):
    """Create summary comparison table."""
    rows = []
    for name, data in datasets.items():
        for split, df in data.items():
            if df.empty:
                continue
            rows.append({
                "Dataset": name,
                "Split": split,
                "Images": len(df),
                "Identities": df["identity_id"].nunique(),
                "Cameras": df["camera_id"].nunique(),
                "Avg Images/ID": f"{len(df) / max(df['identity_id'].nunique(), 1):.1f}",
            })

    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False))
    return summary

summary = summarize_datasets({"Market-1501": market_data, "VeRi-776": veri_data})

## 5. Package Manifests for Download

In [ ]:
import zipfile

# Create a zip of all manifests for easy download
zip_path = Path("/kaggle/working/manifests.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.glob("*.csv"):
        zf.write(f, f.name)
    for f in OUTPUT_DIR.glob("*.png"):
        zf.write(f, f.name)

print(f"\nAll manifests packaged: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024:.1f} KB")
print("\nDownload this file and extract to your local project's data/ directory.")
print("Files included:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  - {f.name}")

## Next Steps
1. **Download** the `manifests.zip` from this notebook's output
2. Extract to your local project: `data/manifests/`
3. Proceed to **Notebook 02** (Person ReID Training) or **Notebook 03** (Vehicle ReID Training)

**Note**: The training notebooks expect the raw dataset images to be available as Kaggle datasets.
Add Market-1501 and VeRi-776 as input datasets when running training notebooks.